In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# 클래식 피드포워드와 제어 흐름 (동적 Circuit)

import Tabs from '@theme/Tabs';
import TabItem from '@theme/TabItem';



# 클래식 피드포워드와 제어 흐름


<details>
<summary><b>패키지 버전</b></summary>

이 페이지의 코드는 다음 요구 사항을 사용하여 개발되었습니다.
아래 버전 이상을 사용하는 것을 권장합니다.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>
> **Note:** 동적 Circuit의 새 버전이 이제 모든 사용자와 모든 Backend에서 사용 가능합니다. 이제 유틸리티 규모로 동적 Circuit을 실행할 수 있습니다. 자세한 내용은 [공지 사항](/announcements/product-updates/2025-09-25-new-dynamic-circuits)을 참조하세요.

동적 Circuit은 강력한 도구로, 양자 Circuit 실행 중에 Qubit을 측정하고 해당 중간 Circuit 측정 결과를 바탕으로 Circuit 내에서 클래식 논리 연산을 수행할 수 있습니다. 이 과정은 _클래식 피드포워드_라고도 알려져 있습니다. 동적 Circuit을 어떻게 활용할지에 대한 이해는 아직 초기 단계이지만, 양자 연구 커뮤니티는 이미 다음과 같은 여러 활용 사례를 발굴했습니다.

* [GHZ 상태,](https://journals.aps.org/prxquantum/abstract/10.1103/PRXQuantum.5.030339) [W-상태](https://arxiv.org/abs/2403.07604) 등 효율적인 양자 상태 준비 (W-상태에 대한 자세한 내용은 ["피드포워드를 사용하는 얕은 Circuit에 의한 상태 준비"](https://arxiv.org/abs/2307.14840)도 참조하세요) 및 광범위한 [행렬 곱 상태](https://arxiv.org/abs/2404.16083)
* 얕은 Circuit을 이용한 같은 칩 위의 Qubit 간 [효율적인 장거리 얽힘](https://journals.aps.org/prxquantum/abstract/10.1103/PRXQuantum.5.030339)
* [IQP 유사 Circuit의 효율적인 샘플링](https://arxiv.org/pdf/2505.04705)

그러나 동적 Circuit이 가져오는 이러한 개선에는 트레이드오프가 있습니다. 중간 Circuit 측정과 클래식 연산은 일반적으로 2-Qubit Gate보다 실행 시간이 더 길며, 이 시간 증가가 Circuit 깊이 감소의 이점을 상쇄할 수 있습니다. 따라서 IBM Quantum&reg;이 동적 Circuit의 [새 버전](/announcements/product-updates/2025-03-03-new-version-dynamic-circuits)을 출시하면서 중간 Circuit 측정 시간 단축이 주요 개선 목표가 되고 있습니다.

[OpenQASM 3 명세](https://openqasm.com/language/classical.html#looping-and-branching)는 다양한 제어 흐름 구조를 정의하고 있지만, Qiskit Runtime은 현재 조건부 `if` 문만 지원합니다. Qiskit SDK에서 이는 [QuantumCircuit](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit)의 [if_test](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#if_test) 메서드에 해당합니다. 이 메서드는 [컨텍스트 매니저](https://docs.python.org/3/reference/datamodel.html#with-statement-context-managers)를 반환하며, 일반적으로 `with` 문에서 사용됩니다. 이 가이드는 이 조건문을 사용하는 방법을 설명합니다.

> **Note:** 이 가이드의 코드 예제는 중간 Circuit 측정에 표준 측정 명령을 사용합니다. 그러나 Backend가 지원하는 경우 [`MidCircuitMeasure`](/guides/measure-qubits#midcircuit) 명령을 대신 사용하는 것을 권장합니다. 자세한 내용은 [중간 Circuit 측정 문서](/guides/measure-qubits#mid-circuit-measurements)를 참조하세요.
## `if` 문
`if` 문은 클래식 비트 또는 레지스터의 값을 기반으로 조건부로 연산을 수행하는 데 사용됩니다.

아래 예제에서는 Qubit에 Hadamard Gate를 적용하고 측정합니다. 결과가 1이면 Qubit에 X Gate를 적용하여 0 상태로 되돌립니다. 그런 다음 Qubit을 다시 측정합니다. 최종 측정 결과는 100% 확률로 0이 됩니다.

In [1]:
from qiskit.circuit import QuantumCircuit, QuantumRegister, ClassicalRegister

qubits = QuantumRegister(1)
clbits = ClassicalRegister(1)
circuit = QuantumCircuit(qubits, clbits)
(q0,) = qubits
(c0,) = clbits

circuit.h(q0)
# Use MidCircuitMeasure() if it's supported by the backend.
# circuit.append(MidCircuitMeasure(), [q0], [c0])
circuit.measure(q0, c0)
with circuit.if_test((c0, 1)):
    circuit.x(q0)
circuit.measure(q0, c0)
circuit.draw("mpl")

# example output counts: {'0': 1024}

<Image src="../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/60924bfa-50ed-4d9d-a17b-9d64f2cc053f-0.svg" alt="Output of the previous code cell" />

![이전 코드 셀의 출력](../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/60924bfa-50ed-4d9d-a17b-9d64f2cc053f-0.svg)

`with` 문에는 컨텍스트 매니저 자체인 할당 대상을 지정할 수 있으며, 이를 저장해 두었다가 나중에 else 블록을 만드는 데 사용할 수 있습니다. else 블록은 `if` 블록의 내용이 *실행되지 않을* 때마다 실행됩니다.

아래 예제에서는 두 Qubit과 두 클래식 비트로 레지스터를 초기화합니다. 첫 번째 Qubit에 Hadamard Gate를 적용하고 측정합니다. 결과가 1이면 두 번째 Qubit에 Hadamard Gate를 적용하고, 그렇지 않으면 두 번째 Qubit에 X Gate를 적용합니다. 마지막으로 두 번째 Qubit도 측정합니다.

In [2]:
qubits = QuantumRegister(2)
clbits = ClassicalRegister(2)
circuit = QuantumCircuit(qubits, clbits)
(q0, q1) = qubits
(c0, c1) = clbits

circuit.h(q0)
circuit.measure(q0, c0)
with circuit.if_test((c0, 1)) as else_:
    circuit.h(q1)
with else_:
    circuit.x(q1)
circuit.measure(q1, c1)

circuit.draw("mpl")

# example output counts: {'01': 260, '11': 272, '10': 492}

<Image src="../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/20f0640a-a3f7-41b3-aada-b66bc89b0555-0.svg" alt="Output of the previous code cell" />

![이전 코드 셀의 출력](../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/20f0640a-a3f7-41b3-aada-b66bc89b0555-0.svg)

단일 클래식 비트에 조건을 거는 것 외에도, 여러 비트로 구성된 클래식 레지스터의 값에 조건을 거는 것도 가능합니다.

아래 예제에서는 두 Qubit에 Hadamard Gate를 적용하고 측정합니다. 결과가 `01`, 즉 첫 번째 Qubit이 1이고 두 번째 Qubit이 0이면, 세 번째 Qubit에 X Gate를 적용합니다. 마지막으로 세 번째 Qubit을 측정합니다. 명확성을 위해 세 번째 클래식 비트의 상태(0)를 `if` 조건에 명시적으로 지정했습니다. Circuit 다이어그램에서 조건은 조건이 걸린 클래식 비트의 원으로 표시됩니다. 검은 원은 1에 대한 조건을, 흰 원은 0에 대한 조건을 나타냅니다.

In [3]:
qubits = QuantumRegister(3)
clbits = ClassicalRegister(3)
circuit = QuantumCircuit(qubits, clbits)
(q0, q1, q2) = qubits
(c0, c1, c2) = clbits

circuit.h([q0, q1])
circuit.measure(q0, c0)
circuit.measure(q1, c1)
with circuit.if_test((clbits, 0b001)):
    circuit.x(q2)
circuit.measure(q2, c2)

circuit.draw("mpl")

# example output counts: {'101': 269, '011': 260, '000': 252, '010': 243}

<Image src="../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/98e8f552-4169-42a3-8182-e14e9ffb59e2-0.svg" alt="Output of the previous code cell" />

![이전 코드 셀의 출력](../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/98e8f552-4169-42a3-8182-e14e9ffb59e2-0.svg)

## 클래식 표현식
Qiskit 클래식 표현식 모듈 [`qiskit.circuit.classical`](https://docs.quantum.ibm.com/api/qiskit/circuit_classical)은 Circuit 실행 중 클래식 값에 대한 런타임 연산의 탐색적 표현을 포함합니다. 하드웨어 제한으로 인해 현재는 `QuantumCircuit.if_test()` 조건만 지원됩니다.

다음 예제는 패리티 계산을 사용하여 동적 Circuit으로 n-Qubit GHZ 상태를 생성하는 방법을 보여줍니다. 먼저 인접한 Qubit에 $n/2$개의 Bell 쌍을 생성합니다. 그런 다음 쌍 사이에 CNOT Gate 레이어를 사용하여 이 쌍들을 연결합니다. 이전 CNOT Gate의 타겟 Qubit을 모두 측정하고 측정된 각 Qubit을 $\vert 0 \rangle$ 상태로 초기화합니다. 이전 모든 비트의 패리티가 홀수인 측정되지 않은 각 사이트에 $X$를 적용합니다. 마지막으로 측정된 Qubit에 CNOT Gate를 적용하여 측정에서 손실된 얽힘을 재확립합니다.

패리티 계산에서 구성된 표현식의 첫 번째 요소는 Python 객체 `mr[0]`을 [`Value`](https://docs.quantum.ibm.com/api/qiskit/circuit_classical#value) 노드로 들어올리는 것입니다 (`lift`는 임의의 객체를 클래식 표현식으로 변환하는 데 사용됩니다). `mr[1]`과 이후의 가능한 클래식 레지스터에는 이것이 필요하지 않습니다. 이들은 `expr.bit_xor`의 입력이며, 필요한 변환은 이 경우에 자동으로 수행됩니다. 이러한 표현식은 루프 및 다른 구조에서 점차적으로 구성할 수 있습니다.

In [ ]:
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.circuit.classical import expr

num_qubits = 8
if num_qubits % 2 or num_qubits < 4:
    raise ValueError("num_qubits must be an even integer ≥ 4")
meas_qubits = list(range(2, num_qubits, 2))  # qubits to measure and reset

qr = QuantumRegister(num_qubits, "qr")
mr = ClassicalRegister(len(meas_qubits), "m")
qc = QuantumCircuit(qr, mr)

# Create local Bell pairs
qc.reset(qr)
qc.h(qr[::2])
for ctrl in range(0, num_qubits, 2):
    qc.cx(qr[ctrl], qr[ctrl + 1])

# Glue neighboring pairs
for ctrl in range(1, num_qubits - 1, 2):
    qc.cx(qr[ctrl], qr[ctrl + 1])

# Measure boundary qubits between pairs,reset to 0
for k, q in enumerate(meas_qubits):
    qc.measure(qr[q], mr[k])
    qc.reset(qr[q])

# Parity-conditioned X corrections
# Each non-measured qubit gets flipped iff the parity (XOR) of all
# preceding measurement bits is 1
for tgt in range(num_qubits):
    if tgt in meas_qubits:  # skip measured qubits
        continue
    # all measurement registers whose physical qubit index < tgt
    left_bits = [k for k, q in enumerate(meas_qubits) if q < tgt]
    if not left_bits:  # skip if list empty
        continue

    # build XOR-parity expression
    parity = expr.lift(
        mr[left_bits[0]]
    )  # lift the first bit to Value so it will be treated like a boolean.
    for k in left_bits[1:]:
        parity = expr.bit_xor(
            mr[k], parity
        )  # calculate parity with all other bits
    with qc.if_test(parity):  # Add X if parity is 1
        qc.x(qr[tgt])

# Re-entangle measured qubits
for ctrl in range(1, num_qubits - 1, 2):
    qc.cx(qr[ctrl], qr[ctrl + 1])

In [5]:
qc.draw(output="mpl", style="iqp", idle_wires=False, fold=-1)

<Image src="../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/d0f0abdb-50d5-408d-a704-a1a555acdd85-0.svg" alt="Output of the previous code cell" />

![이전 코드 셀의 출력](../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/d0f0abdb-50d5-408d-a704-a1a555acdd85-0.svg)

## 동적 Circuit을 지원하는 Backend 찾기
계정에서 액세스할 수 있고 동적 Circuit을 지원하는 모든 Backend를 찾으려면 다음과 같은 코드를 실행하세요. 이 예제는 [로그인 자격 증명을 저장](/guides/save-credentials)했다고 가정합니다. Qiskit Runtime 서비스 계정을 초기화할 때 [명시적으로 자격 증명을 지정](/guides/initialize-account#explicit)할 수도 있습니다. 예를 들어 특정 인스턴스나 플랜 유형에서 사용 가능한 Backend를 볼 수 있습니다.

> **Note:** - 계정에서 사용 가능한 Backend는 자격 증명에 지정된 인스턴스에 따라 다릅니다.
> - 동적 Circuit의 새 버전이 이제 모든 사용자와 모든 Backend에서 사용 가능합니다. 자세한 내용은 [공지 사항](/announcements/product-updates/2025-09-25-new-dynamic-circuits)을 참조하세요.

In [6]:
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
dc_backends = service.backends(dynamic_circuits=True)
print(dc_backends)

[<IBMBackend('ibm_pittsburgh')>, <IBMBackend('ibm_boston')>, <IBMBackend('ibm_fez')>, <IBMBackend('ibm_miami')>, <IBMBackend('ibm_marrakesh')>, <IBMBackend('ibm_torino')>, <IBMBackend('ibm_kingston')>]


## Qiskit Runtime limitations

Be aware of the following constraints when running dynamic circuits in Qiskit Runtime.

- Due to the limited physical memory on control electronics, there is also a limit on the number of `if` statements and size of their operands. This limit is a function of the number of broadcasts and the number of broadcasted bits in a job (not a circuit).

   When processing an `if` condition, measurement data needs to be transferred to the control logic to make that evaluation. A broadcast is a transfer of unique classical data, and broadcasted bits is the number of classical bits being transferred. Consider the following:

   ```python
   c0 = ClassicalRegister(3)
   c1 = ClassicalRegister(5)
   ...
   with circuit.if_test((c0, 1)) ...
   with circuit.if_test((c0, 3)) ...
   with circuit.if_test((c1[2], 1)) ...
   ```
   In the previous code example, the first two `if_test` objects on `c0` are considered one broadcast because the content of `c0` did not change, and thus does not need to be re-broadcasted. The `if_test` on `c1` is a second broadcast. The first one broadcasts all three bits in `c0` and the second broadcasts just one bit, making a total of four broadcasted bits.

   Currently, if you broadcast 60 bits each time, then the job can have approximately 300 broadcasts. If you broadcast just one bit each time, however, then the job can have 2400 broadcasts.

- The operand used in an `if_test` statement must be 32 or fewer bits. Thus, if you are comparing an entire `ClassicalRegister`, the size of that `ClassicalRegister` must be 32 or fewer bits. If you are comparing just a single bit from a `ClassicalRegister`, however, that `ClassicalRegister` can be of any size (since the operand is only one bit).

   For example, the "Not valid" code block does not work because `cr` is more than 32 bits.  You can, however, use a classical register wider than 32 bits if you are testing only one bit, as shown in the "Valid" code block.

   <Tabs>
   <TabItem value="Not valid" label="Not valid">
      ```python
         cr = ClassicalRegister(50)
         qr = QuantumRegister(50)
         circuit = QuantumCircuit(qr, cr)
         ...
         circ.measure(qr, cr)
         with circ.if_test((cr, 15)):
            ...
      ```
   </TabItem>
   <TabItem value="Valid" label="Valid">
      ```python
         cr = ClassicalRegister(50)
         qr = QuantumRegister(50)
         circuit = QuantumCircuit(qr, cr)
         ...
         circ.measure(qr, cr)
         with circ.if_test((cr[5], 1)):
            ...
      ```
   </TabItem>
   </Tabs>

- Nested conditionals are not allowed. For example, the following code block will not work because it has an `if_test` inside another `if_test`:
   <Tabs>
    <TabItem value="Not valid" label="Not valid">
        ```python
           c1 = ClassicalRegister(1, "c1")
           c2 = ClassicalRegister(2, "c2")
           ...
           with circ.if_test((c1, 1)):
            with circ.if_test(c2, 1)):
             ...
        ```
     </TabItem>
     <TabItem value="Valid" label="Valid">
        ```python
        cr = ClassicalRegister(2)
        ...
        with circuit.if_test((cr, 0b11)):
          ...
        ```
     </TabItem>
    </Tabs>

- Having `reset` or measurements inside conditionals is not supported.
- Arithmetic operations are not supported.
- See the [OpenQASM 3 feature table](/docs/guides/qasm-feature-table) to determine which OpenQASM 3 features are supported on Qiskit and Qiskit Runtime.
- When OpenQASM 3 (instead of `QuantumCircuit`) is used as the input format to pass circuits to Qiskit Runtime primitives, only instructions that can be loaded into Qiskit are supported. Classical operations, for example, are not supported because they cannot be loaded into Qiskit. See [Import an OpenQASM 3 program into Qiskit](/docs/guides/interoperate-qiskit-qasm3#import-an-openqasm-3-program-into-qiskit) for more information.
- The `for`, `while`, and `switch` instructions are not supported.

## Use dynamic circuits with Estimator

Since Estimator does not support dynamic circuits, you can use Sampler and build your own measurement circuits instead. Alternatively, you can use the [Executor primitive,](/docs/guides/directed-execution-model#executor-primitive) which supports dynamic circuits.

To replicate Estimator's behavior, follow this process:

1. Group the terms of all observables into a partition.  This can be done by using the [`PauliList` API,](/docs/api/qiskit/qiskit.quantum_info.PauliList#group_qubit_wise_commuting) for example.
     <Admonition type="note">
      You can use the [`BitArray`](https://quantum.cloud.ibm.com/docs/en/api/qiskit/qiskit.primitives.BitArray#expectation_values) primitive attribute to compute the expectation values of the provided observables.
     </Admonition>
2. Execute one basis change circuit per partition (whichever basis change needs to be done for each partition). See the Measurement bases addon utility  [`measurement_bases` module](https://github.com/Qiskit/qiskit-addon-utils/blob/38ea05431f2e9830adf4ec9265f6d19758a32096/qiskit_addon_utils/exp_vals/measurement_bases.py) for more information. [Get started with utilities.](/docs/guides/qiskit-addons-utils#get-started-with-utilities)
3. Add back together the results for each partition.

## Next steps

<Admonition type="tip" title="Recommendations">
- Learn how to implement accurate dynamic decoupling by using [stretch.](/docs/guides/stretch)
- Learn about the shorter [mid-circuit measurements](/docs/guides/measure-qubits#mid-circuit-measurements) that reduce the circuit time.
- Use [circuit schedule visualization](/docs/guides/visualize-circuit-timing#qiskit-runtime-support) to debug and optimize your dynamic circuits.
</Admonition>